# 37. 모음조화 & 모음충돌회피

v7 사전의 용언(동사/형용사)에서 모음조화와 모음충돌회피 환경을 추출

## 배경 (10강 세미나 자료)

### 모음조화 (2강)
- 양성모음(ㅏ,ㅗ) 어간 + -아/아서 (ex: 잡다→잡아, 좋다→좋아)
- 음성모음(ㅓ,ㅡ,ㅣ...) 어간 + -어/어서 (ex: 먹다→먹어, 쓰다→써)

### 모음충돌회피 (10강)
- 모음어간 + 모음어미 → 회피 전략:
  1. **탈락**: ㅓ+ㅓ → ㅓ (떼+어 → 떼)
  2. **활음화**: i+ㅓ → jㅓ (끼+어 → 껴), o+ㅓ → wㅓ (보+아 → 봐)
  3. **활음삽입**: i+ㅓ → ijㅓ (끼+어 → 끼여), o+ㅓ → owㅓ (보+아 → 보와)
- 변이 요인: 어간모음 종류, 음절수, 발화속도, 빈도

## 접근방법
1. v7에서 동사/형용사 추출
2. conjugations 필드에서 -아/어 활용형 파싱
3. 어간 마지막 모음 분석 → 모음조화/충돌회피 환경 분류
4. 후보 리스트 CSV 저장
5. → 50/60번에서 코퍼스 검색

## 1. 환경 설정

In [1]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

V7_LEXICON = f'{PROJECT_ROOT}/10_dictionary_build/output/04_v7_lexicon.csv'
RESULT_DIR = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

## 2. 데이터 로드 및 용언 필터링

In [3]:
print('v7 lexicon 로딩...')
df = pd.read_csv(V7_LEXICON, encoding='utf-8-sig', low_memory=False)
print(f'v7: {len(df):,}행')

# 동사 + 형용사 필터
df_verb = df[df['pos'].isin(['동사', '형용사'])].copy()
print(f'\n용언: {len(df_verb):,}행')
print(f'  동사: {(df_verb["pos"]=="동사").sum():,}')
print(f'  형용사: {(df_verb["pos"]=="형용사").sum():,}')

# conjugations 필드 유무 확인
has_conj = df_verb['conjugations'].notna().sum()
print(f'  conjugations 있음: {has_conj:,}')
print(f'  conjugations 없음: {len(df_verb) - has_conj:,}')

# 샘플
print('\n샘플 (conjugations):')
for _, r in df_verb[df_verb['conjugations'].notna()].head(5).iterrows():
    print(f'  {r["word"]} ({r["pos"]}): {str(r["conjugations"])[:80]}')

v7 lexicon 로딩...
v7: 528,088행

용언: 96,386행
  동사: 78,792
  형용사: 17,594
  conjugations 있음: 93,413
  conjugations 없음: 2,973

샘플 (conjugations):
  겅둥겅둥하다 (동사): 겅둥겅둥하여,겅둥겅둥해,겅둥겅둥하니
  겅둥겅둥하다 (동사): 겅둥겅둥하여,겅둥겅둥해,겅둥겅둥하니
  겅둥대다 (동사): 겅둥대어,겅둥대,겅둥대니
  겅둥대다 (동사): 겅둥대어,겅둥대,겅둥대니
  겅둥하다 (형용사): 겅둥하여,겅둥해,겅둥하니


## 3. 한글 모음 분석 함수

In [ ]:
# 한글 자모 분해
def decompose_hangul(char):
    """한글 1글자 -> (초성, 중성, 종성) 인덱스"""
    if not char or len(char) != 1:
        return None
    code = ord(char) - 0xAC00
    if code < 0 or code > 11171:
        return None
    cho = code // (21 * 28)
    jung = (code % (21 * 28)) // 28
    jong = code % 28
    return cho, jung, jong

# 중성(모음) 인덱스 -> 모음 문자
JUNGSEONG = [
    'ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ',
    'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ',
    'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ'
]

# 종성 인덱스 -> 종성 문자 (0=없음)
JONGSEONG = [
    '', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ',
    'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ',
    'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
]

# 양성모음 vs 음성모음
POSITIVE_VOWELS = {'ㅏ', 'ㅗ'}          # 양성
NEGATIVE_VOWELS = {'ㅓ', 'ㅕ', 'ㅛ', 'ㅡ', 'ㅣ'}  # 음성

def get_stem_final_vowel(stem):
    """어간의 마지막 모음 추출"""
    if not stem:
        return None, None
    for ch in reversed(stem):
        d = decompose_hangul(ch)
        if d:
            jung_idx = d[1]
            jong = d[2]
            vowel = JUNGSEONG[jung_idx]
            return vowel, jong == 0  # (모음, 개음절 여부)
    return None, None

def classify_vowel_polarity(vowel):
    """모음 양/음성 분류"""
    if vowel in POSITIVE_VOWELS:
        return 'positive'  # 양성 (ㅏ,ㅗ)
    elif vowel in NEGATIVE_VOWELS:
        return 'negative'  # 음성
    else:
        # 복합모음: ㅐ(애)=양성, ㅘ(와)=양성 등
        positive_compound = {'ㅐ', 'ㅘ', 'ㅙ', 'ㅚ'}
        if vowel in positive_compound:
            return 'positive'
        return 'negative'

def get_vowel_name(vowel):
    """모음 기호 -> 이름"""
    return vowel if vowel else ''

# --- 신규 함수: 세미나 2강 기반 변이 요인 ---

def get_stem_syllable_count(stem):
    """어간 음절수 (세미나 2강: 단음절 > 다음절)"""
    if not stem:
        return 0
    count = 0
    for ch in stem:
        d = decompose_hangul(ch)
        if d is not None:
            count += 1
    return count

def get_stem_final_coda(stem):
    """
    어간말 종성 정보 추출

    Returns: (종성문자, 종성수, 종성유형)
      종성문자: 'ㄱ', 'ㄹㅂ' 등
      종성수: 0 (개음절), 1 (단자음), 2 (겹자음)
      종성유형: 'none' / 'obstruent' / 'sonorant' / 'cluster_obs' / 'cluster_son' / 'h'
    """
    if not stem:
        return '', 0, 'none'
    for ch in reversed(stem):
        d = decompose_hangul(ch)
        if d is not None:
            jong_idx = d[2]
            if jong_idx == 0:
                return '', 0, 'none'
            jong_char = JONGSEONG[jong_idx]

            # 겹자음 판별
            cluster_map = {
                'ㄳ': ('ㄱㅅ', 2), 'ㄵ': ('ㄴㅈ', 2), 'ㄶ': ('ㄴㅎ', 2),
                'ㄺ': ('ㄹㄱ', 2), 'ㄻ': ('ㄹㅁ', 2), 'ㄼ': ('ㄹㅂ', 2),
                'ㄽ': ('ㄹㅅ', 2), 'ㄾ': ('ㄹㅌ', 2), 'ㄿ': ('ㄹㅍ', 2),
                'ㅀ': ('ㄹㅎ', 2), 'ㅄ': ('ㅂㅅ', 2),
            }
            if jong_char in cluster_map:
                expanded, count = cluster_map[jong_char]
                # 겹자음 유형: 마지막 자음 기준
                last_c = expanded[-1]
                if last_c in ('ㅎ',):
                    coda_type = 'cluster_h'
                elif last_c in ('ㄱ','ㄷ','ㅂ','ㅅ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ'):
                    coda_type = 'cluster_obs'
                else:
                    coda_type = 'cluster_son'
                return expanded, count, coda_type

            # 단자음
            if jong_char == 'ㅎ':
                coda_type = 'h'
            elif jong_char in ('ㄱ','ㄲ','ㄷ','ㅂ','ㅅ','ㅆ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ'):
                coda_type = 'obstruent'
            elif jong_char in ('ㄴ','ㅁ','ㅇ','ㄹ'):
                coda_type = 'sonorant'
            else:
                coda_type = 'obstruent'
            return jong_char, 1, coda_type
    return '', 0, 'none'

def detect_irregular_type(word, stem, conj_form):
    """
    어간과 활용형을 비교하여 불규칙 용언 유형 추정

    Returns: 'regular' / 'p_irregular' / 'h_irregular' / 'd_irregular' /
             'l_irregular' / 's_irregular' / 'ru_irregular' / 'unknown_irregular'

    판별 방식:
    - word가 '~다'로 끝나는 것을 전제로, 어간(word[:-1])의 끝 자음과
      활용형의 패턴을 비교하여 추정
    """
    if not word or not stem or not conj_form:
        return ''

    # 어간말 종성
    d = None
    for ch in reversed(stem):
        d = decompose_hangul(ch)
        if d is not None:
            break
    if d is None:
        return ''

    jong_idx = d[2]
    jong = JONGSEONG[jong_idx] if jong_idx > 0 else ''

    # 개음절 어간 → 불규칙 해당 없거나 별도 처리
    if jong == '':
        # 르 불규칙: 어간이 '르'로 끝나는 경우
        if stem.endswith('르') and len(conj_form) >= 2:
            # 자르다 → 잘라 (르 → ㄹ+라)
            for ch in conj_form:
                dd = decompose_hangul(ch)
                if dd and JONGSEONG[dd[2]] == 'ㄹ':
                    return 'ru_irregular'
        return 'regular'

    # ㅂ 불규칙: 어간 끝 ㅂ → 활용형에 우/워 포함
    if jong == 'ㅂ':
        for ch in conj_form:
            dd = decompose_hangul(ch)
            if dd:
                v = JUNGSEONG[dd[1]]
                if v in ('ㅝ', 'ㅘ', 'ㅜ'):
                    return 'p_irregular'
        # ㅂ이 유지되면 규칙
        return 'regular'

    # ㄷ 불규칙: 어간 끝 ㄷ → 활용형에 ㄹ 종성
    if jong == 'ㄷ':
        # 걷다→걸어: 어간 '걷' 활용 '걸어'
        if len(conj_form) >= 1:
            dd = decompose_hangul(conj_form[-1] if len(conj_form) == len(stem) else
                                  conj_form[len(stem)-1] if len(conj_form) >= len(stem) else '')
            if dd and JONGSEONG[dd[2]] == 'ㄹ':
                return 'd_irregular'
            # 더 안전한 방법: 어간 마지막 글자 위치에 ㄹ 종성이 오면
            if len(conj_form) >= len(stem):
                check_ch = conj_form[len(stem)-1]
                dd2 = decompose_hangul(check_ch)
                if dd2 and JONGSEONG[dd2[2]] == 'ㄹ':
                    return 'd_irregular'
        return 'regular'

    # ㅎ 불규칙: 어간 끝 ㅎ → 활용형에서 ㅎ 탈락
    if jong == 'ㅎ':
        if len(conj_form) >= len(stem):
            check_ch = conj_form[len(stem)-1]
            dd = decompose_hangul(check_ch)
            if dd and JONGSEONG[dd[2]] == '':
                return 'h_irregular'
        return 'regular'

    # ㅅ 불규칙: 어간 끝 ㅅ → 활용형에서 ㅅ 탈락
    if jong == 'ㅅ':
        if len(conj_form) >= len(stem):
            check_ch = conj_form[len(stem)-1]
            dd = decompose_hangul(check_ch)
            if dd and JONGSEONG[dd[2]] == '':
                return 's_irregular'
        return 'regular'

    # ㄹ 탈락 (ㄹ 불규칙이 아닌 ㄹ 탈락): 알다→아는
    # 이건 -아/-어 활용에서는 해당 안됨

    return 'regular'

# 테스트
tests = [('잡', 'ㅏ', False), ('먹', 'ㅓ', False), ('가', 'ㅏ', True), ('쓰', 'ㅡ', True)]
for stem, exp_v, exp_open in tests:
    v, is_open = get_stem_final_vowel(stem)
    ok = (get_vowel_name(v) == get_vowel_name(exp_v)) and (is_open == exp_open)
    print(f'  {stem}: 모음={get_vowel_name(v)}, 개음절={is_open} {"OK" if ok else "FAIL"}')

# 신규 함수 테스트
print('\n--- 어간 음절수 ---')
for s in ['잡', '먹', '따르', '겅둥겅둥하']:
    print(f'  {s}: {get_stem_syllable_count(s)}음절')

print('\n--- 어간말 종성 ---')
for s in ['잡', '읽', '낳', '가', '맑', '없']:
    coda, cnt, ctype = get_stem_final_coda(s)
    print(f'  {s}: 종성={coda}, 수={cnt}, 유형={ctype}')

print('\n--- 불규칙 감지 ---')
irr_tests = [
    ('잡다', '잡', '잡아', 'regular'),
    ('돕다', '돕', '도와', 'p_irregular'),
    ('걷다', '걷', '걸어', 'd_irregular'),
    ('낫다', '낫', '나아', 's_irregular'),
    ('노랗다', '노랗', '노래', 'h_irregular'),
]
for w, s, c, exp in irr_tests:
    result = detect_irregular_type(w, s, c)
    print(f'  {w}: {result} {"OK" if result == exp else "FAIL (expected " + exp + ")"}')

## 4. conjugations 파싱

In [ ]:
def parse_conjugations(conj_str):
    """
    conjugations 필드에서 -아/어 활용형 추출

    v7 conjugations 형식: "잡아,잡으니,잡는" 또는 "가,가니"
    첫 번째 항목이 -아/어 활용형
    """
    if pd.isna(conj_str) or not conj_str:
        return None

    conj = str(conj_str).strip().strip('"').strip()
    if not conj:
        return None

    parts = conj.split(',')
    if parts:
        return parts[0].strip()
    return None

# 테스트
test_cases = [
    ('잡아,잡으니,잡는', '잡아'),
    ('가,가니', '가'),
    ('와,오니', '와'),
    ('써,쓰니', '써'),
]
for conj, expected in test_cases:
    result = parse_conjugations(conj)
    print(f'  "{conj}" -> "{result}" {"OK" if result == expected else "FAIL"}')

## 5. 모음조화 & 충돌회피 환경 분류

In [ ]:
def classify_vowel_environment(word_stem, conj_form, word):
    """
    어간+활용형을 분석하여 모음조화/충돌회피 환경 분류

    Returns: dict with environment info
    """
    result = {
        'stem': word_stem if pd.notna(word_stem) else word.rstrip('다') if word else '',
        'conj_form': conj_form,
        'stem_final_vowel': None,
        'stem_final_vowel_name': '',
        'stem_ends_open': None,  # 개음절 여부
        'vowel_polarity': '',     # positive/negative
        'suffix_type': '',        # -아/-어
        'harmony_ok': None,       # 모음조화 일치 여부
        'collision_type': '',     # deletion/glide_formation/glide_insertion/none
        'environment': '',        # vowel_harmony / vowel_collision / both
    }

    stem = result['stem']
    if not stem or not conj_form:
        return result

    # 어간 마지막 모음
    vowel, is_open = get_stem_final_vowel(stem)
    if vowel is None:
        return result

    result['stem_final_vowel'] = vowel
    result['stem_final_vowel_name'] = get_vowel_name(vowel)
    result['stem_ends_open'] = is_open
    result['vowel_polarity'] = classify_vowel_polarity(vowel)

    # -아/어 판별: 활용형에서 어미 부분 추정
    if len(conj_form) > len(stem):
        suffix_part = conj_form[len(stem):]
        if suffix_part and suffix_part[0] in '아':
            result['suffix_type'] = '-아'
        elif suffix_part and suffix_part[0] in '어여':
            result['suffix_type'] = '-어'
    elif len(conj_form) <= len(stem):
        # 탈락/축약 발생 (ex: 가+아 -> 가, 서+어 -> 서)
        pass

    # 모음조화 일치 판별
    if result['suffix_type']:
        if result['vowel_polarity'] == 'positive' and result['suffix_type'] == '-아':
            result['harmony_ok'] = True
        elif result['vowel_polarity'] == 'negative' and result['suffix_type'] == '-어':
            result['harmony_ok'] = True
        else:
            result['harmony_ok'] = False

    # 모음충돌회피 판별 (어간이 모음으로 끝날 때만)
    if is_open:
        result['environment'] = 'vowel_collision'

        if len(conj_form) < len(stem):
            result['collision_type'] = 'contraction'  # 축약 (오+아 -> 와)
        elif len(conj_form) == len(stem):
            if conj_form == stem:
                result['collision_type'] = 'deletion'  # 동일모음 탈락
            else:
                result['collision_type'] = 'glide_formation'  # 활음화
        elif len(conj_form) == len(stem) + 1:
            result['collision_type'] = 'glide_insertion'  # 활음삽입 (보+아 -> 보와)
        else:
            result['collision_type'] = 'other'
    else:
        result['environment'] = 'vowel_harmony'  # 자음어간 -> 모음조화만
        result['collision_type'] = 'none'

    return result

# 테스트
test_envs = [
    ('잡', '잡아', '잡다'),    # 자음어간 + -아 (모음조화)
    ('먹', '먹어', '먹다'),    # 자음어간 + -어 (모음조화)
    ('가', '가', '가다'),      # 모음어간 -> 탈락 (모음충돌)
    ('오', '와', '오다'),      # 모음어간 -> 활음화 (모음충돌)
    ('쓰', '써', '쓰다'),      # ㅡ어간 -> 축약
]
for stem, conj, word in test_envs:
    r = classify_vowel_environment(stem, conj, word)
    print(f'  {word}: 어간={stem}, 활용={conj}, '
          f'모음={r["stem_final_vowel_name"]}, 양음성={r["vowel_polarity"]}, '
          f'환경={r["environment"]}, 충돌회피={r["collision_type"]}')

## 6. 전체 용언 환경 분류 실행

In [ ]:
# conjugations 파싱 + 환경 분류
results = []

for _, row in df_verb.iterrows():
    conj_form = parse_conjugations(row.get('conjugations', ''))
    if not conj_form:
        continue

    word = str(row.get('word', ''))
    word_stem = row.get('word_stem', '')

    env = classify_vowel_environment(word_stem, conj_form, word)

    # 기본 정보 추가
    env['word'] = word
    env['pos'] = row.get('pos', '')
    env['sense_no'] = row.get('sense_no', '')
    env['word_roman'] = row.get('word_roman', '')
    env['pron_roman'] = row.get('pron_roman', '')
    env['word_type'] = row.get('word_type', '')
    env['conjugations'] = str(row.get('conjugations', ''))[:100]
    env['definition'] = str(row.get('definition', ''))[:200]
    env['freq_LS_total'] = row.get('freq_LS_total', 0)
    env['freq_MP_total'] = row.get('freq_MP_total', 0)

    # --- 신규 조건 변수 (세미나 2강/10강 기반) ---
    stem = env['stem']

    # 어간 음절수 (세미나 2강: 단음절 vs 다음절 변이)
    env['stem_syllable_count'] = get_stem_syllable_count(stem)

    # 어간말 종성 정보 (세미나 10강: 개재 자음 수/질)
    coda_char, coda_count, coda_type = get_stem_final_coda(stem)
    env['coda_char'] = coda_char
    env['coda_count'] = coda_count
    env['coda_type'] = coda_type

    # 불규칙 용언 유형 (세미나 2강: 불규칙 활용 구분)
    env['irregular_type'] = detect_irregular_type(word, stem, conj_form)

    results.append(env)

df_result = pd.DataFrame(results)
print(f'\n전체 분류 결과: {len(df_result):,}행')

print(f'\n환경 분포:')
print(df_result['environment'].value_counts())

print(f'\n충돌회피 유형:')
print(df_result['collision_type'].value_counts())

print(f'\n어간마지막모음:')
print(df_result['stem_final_vowel_name'].value_counts().head(10))

print(f'\n모음조화 일치/불일치:')
print(df_result['harmony_ok'].value_counts())

# 신규 컬럼 확인
print(f'\n--- 신규 컬럼 ---')
print(f'어간 음절수 분포:')
print(df_result['stem_syllable_count'].value_counts().sort_index().head(10))

print(f'\n종성 유형 분포:')
print(df_result['coda_type'].value_counts())

print(f'\n불규칙 유형 분포:')
print(df_result['irregular_type'].value_counts())

## 7. 모음충돌회피 후보 상세 분석

In [ ]:
# 모음충돌회피 후보만 추출
df_collision = df_result[df_result['environment'] == 'vowel_collision'].copy()
print(f'모음충돌회피 후보: {len(df_collision):,}행')

# 어간모음별 회피 전략 분포
print('\n어간모음별 회피 전략:')
ct = pd.crosstab(df_collision['stem_final_vowel_name'], df_collision['collision_type'])
print(ct)

# 상위 20개 예시
print('\n모음충돌회피 예시 (빈도순):')
display_cols = ['word', 'stem', 'conj_form', 'stem_final_vowel_name', 'collision_type', 'freq_LS_total']
print(df_collision.sort_values('freq_LS_total', ascending=False)[display_cols].head(20).to_string())

## 8. 모음조화 후보 상세 분석

In [ ]:
# 모음조화 후보 (자음어간만)
df_harmony = df_result[df_result['environment'] == 'vowel_harmony'].copy()
print(f'모음조화 후보: {len(df_harmony):,}행')

# 양성/음성별 -아/-어 분포
print('\n양음성 x 접미사:')
ct2 = pd.crosstab(df_harmony['vowel_polarity'], df_harmony['suffix_type'])
print(ct2)

# 모음조화 불일치 사례
mismatch = df_harmony[df_harmony['harmony_ok'] == False]
print(f'\n모음조화 불일치: {len(mismatch):,}행')
if len(mismatch) > 0:
    display_cols = ['word', 'stem', 'conj_form', 'stem_final_vowel_name', 'vowel_polarity', 'suffix_type', 'freq_LS_total']
    print(mismatch.sort_values('freq_LS_total', ascending=False)[display_cols].head(20).to_string())

## 9. 결과 CSV 저장

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

column_order = [
    'word', 'pos', 'stem', 'conj_form',
    'environment',  # vowel_harmony / vowel_collision
    'stem_final_vowel_name', 'stem_ends_open', 'vowel_polarity',
    'suffix_type', 'harmony_ok',
    'collision_type',  # deletion/glide_formation/glide_insertion/contraction/none
    # 신규 조건 변수 (세미나 2강/10강)
    'stem_syllable_count',  # 어간 음절수
    'coda_char', 'coda_count', 'coda_type',  # 어간말 종성 정보
    'irregular_type',  # 불규칙 유형
    'word_type', 'word_roman', 'pron_roman',
    'conjugations',
    'freq_LS_total', 'freq_MP_total',
    'sense_no', 'definition',
]

# 전체 저장
out_all = f'{RESULT_DIR}/vowel_harmony_collision_all_{timestamp}.csv'
cols_exist = [c for c in column_order if c in df_result.columns]
df_result[cols_exist].to_csv(out_all, index=False, encoding='utf-8-sig')
print(f'전체: {out_all} ({len(df_result):,}행)')

# 모음충돌회피만 별도 저장
out_collision = f'{RESULT_DIR}/vowel_collision_candidates_{timestamp}.csv'
df_collision[cols_exist].to_csv(out_collision, index=False, encoding='utf-8-sig')
print(f'충돌회피: {out_collision} ({len(df_collision):,}행)')

# 모음조화 불일치만 별도 저장
if len(mismatch) > 0:
    out_mismatch = f'{RESULT_DIR}/vowel_harmony_mismatch_{timestamp}.csv'
    mismatch[cols_exist].to_csv(out_mismatch, index=False, encoding='utf-8-sig')
    print(f'조화불일치: {out_mismatch} ({len(mismatch):,}행)')

print('\n저장 완료')
print('\n다음 단계: 50번(서울코퍼스), 60번(대화코퍼스)에서 이 단어들의 실제 발음 검색')

## 10. 다음 단계: 코퍼스 검색

이 노트북에서 추출한 **어간 리스트**를 50/60번에서 검색:

### 50번 서울코퍼스
- pWord에서 어간+어미 결합 형태 찾기
- ortho vs prono 비교: 실제 발음에서 활음화 vs 탈락 vs 활음삽입 어떤 것이 실현되는지
- 화자별(연령/성별) 변이 통계

### 60번 대화코퍼스
- 자연 대화에서 모음충돌회피 변이 확인
- 발화속도, 화자 변수와의 상관관계
- 수업자료(홍석우 2023, 신우봉 2013)의 변이 요인 검증